# EI across the 8 'P' seizures — Bella, side by side with neural fragility

Runs the repo's own EI (`app.sigproc.ei`, after the 2026-08-10 fixes) over
every `SZ nP` seizure and compares it with the EZFragility result.

**Read the cutoff sweep, not a single table.** Aggregating "top N contacts per
seizure" into shaft votes gives a different winner depending on N — shaft I at
N=5 and N=10, shaft D at N=20 and N=30. Any single choice of N is an argument,
not a result. The cutoff-free comparison is "which shaft wins each seizure
outright", printed below.

**Kernel:** any environment with `mne`, `numpy` and `scipy` — the repo-root
`.venv` is enough. `app/sigproc/` is deliberately free of `sqlalchemy`,
`pydantic` and `sklearn`, so the numerics import without the server stack.

**Inputs:** `data/Bella/BellaEDF/*.edf`, from
`v2/tools/nk2edf/nk2edf.py DA6465AU.EEG <outdir> --blocks 17,20,21,22,26,27,29,44`.
Those carry the `.LOG` events as EDF+ annotations, which is where the `SZ nP`
labels come from.

In [1]:
import glob
import logging
import os
import re
import sys
from collections import defaultdict

import numpy as np

sys.path.insert(0, os.path.abspath("../server"))

from app.sigproc.channels import load_seeg
from app.sigproc.ei import compute_ei_index, compute_hfer
from app.sigproc.filters import filter_for_display

logging.basicConfig(level=logging.ERROR)

EDF_DIR = "../../data/Bella/BellaEDF"

# Every clip marks its seizure at t=120 s, so one window convention serves all 8.
# These are the clip-17 parameters from docs/bella_ictal_ei_vs_annotation_discrepancy.md.
BASELINE = (20.0, 75.0)
TARGET = (100.0, 170.0)
BAND = (1.0, 300.0)
MAINS = 60.0
PAD = 10.0  # filtfilt runway outside the analysed windows

In [2]:
def shaft_of(contact):
    """'X\'12' -> \"X'\", 'A7' -> 'A'. The prime marks the contralateral shaft."""
    return re.match(r"^([A-Za-z]'?)", contact).group(1)


def ei_for_file(path):
    """(label, contacts ranked best-first, EI in that order). Mirrors run_ei_compute_job."""
    raw = load_seeg(path)  # drops REF/DC/EKG/UNUSED/MARK
    fs = raw.info["sfreq"]
    chn = raw.ch_names
    label = next((d.strip() for d in raw.annotations.description
                  if re.match(r"^SZ \d+P$", d.strip())), None)
    duration = float(raw.times[-1])

    span0 = max(0.0, min(BASELINE[0], TARGET[0]) - PAD)
    span1 = min(duration, max(BASELINE[1], TARGET[1]) + PAD)
    raw.crop(tmin=span0, tmax=span1).load_data()
    data, _ = raw[:]
    filtered = filter_for_display(data, fs, *BAND, mains_freq=MAINS)

    def idx(t):
        return int(round((t - span0) * fs))

    norm_t, norm_b = compute_hfer(filtered[:, idx(TARGET[0]):idx(TARGET[1])],
                                  filtered[:, idx(BASELINE[0]):idx(BASELINE[1])], fs)
    ei, _ei_raw, _hfer, _time_coef = compute_ei_index(norm_t, norm_b, fs)
    order = np.argsort(ei)[::-1]
    return label, [chn[c] for c in order], ei[order]

In [3]:
ei_full = {}    # label -> every contact, best first
ei_value = {}   # label -> EI in that same order
shaft_size = {}

for path in sorted(glob.glob(os.path.join(EDF_DIR, "*.edf"))):
    label, ranking, values = ei_for_file(path)
    if label is None:
        print(f"skipping {os.path.basename(path)}: no 'SZ nP' annotation")
        continue
    ei_full[label], ei_value[label] = ranking, values
    if not shaft_size:
        for c in ranking:
            shaft_size[shaft_of(c)] = shaft_size.get(shaft_of(c), 0) + 1
    print(f"{label}: {len(ranking)} contacts ranked")

N_CONTACTS = sum(shaft_size.values())
print(f"\n{N_CONTACTS} contacts across {len(shaft_size)} shafts")

Extracting EDF parameters from ../../data/Bella/BellaEDF/DA6465AU_17_20240319072231.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 170000  =      0.000 ...   170.000 secs...
SZ 1P: 184 contacts ranked
Extracting EDF parameters from ../../data/Bella/BellaEDF/DA6465AU_20_20240319173404.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 170000  =      0.000 ...   170.000 secs...
SZ 2P: 184 contacts ranked
Extracting EDF parameters from ../../data/Bella/BellaEDF/DA6465AU_21_20240319221721.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 170000  =      0.000 ...   170.000 secs...
SZ 3P: 184 contacts ranked
Extracting EDF parameters from ../../data/Bella/BellaEDF/DA6465AU_22_20240320060204.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 170000  =      0.000 ...   170.000 secs...
SZ 4P: 184 contacts ranked
Extracting EDF parameters from ../../dat

## Fragility reference

From `data/ezfragility_result.txt` (EZFragility, R). It analysed a very different
span — 5 s before to 3 s after onset, against EI's 70 s target window — so the two
are not measuring the same event. Its vote counts use its own scheme; only the
ordering is comparable.

In [4]:
FRAG_TOP = {
    "SZ 1P": ["X12", "P1", "D6", "L'11", "D2", "L'8", "M5", "D1", "L'7", "P6"],
    "SZ 2P": ["D2", "D1", "S4", "L8", "D3", "B4", "I3", "F10", "X12", "F6"],
    "SZ 3P": ["S4", "D2", "P6", "T3", "D1", "M5", "P1", "T2", "X12", "K7"],
    "SZ 4P": ["D6", "D2", "D1", "F5", "M5", "A1", "K8", "M8", "F6", "P6"],
    "SZ 5P": ["D1", "D2", "D6", "X'12", "I1", "T3", "T2", "I2", "A1", "X9"],
    "SZ 6P": ["D6", "I1", "M5", "X'12", "I2", "B4", "I3", "I6", "M'10", "X'8"],
    "SZ 7P": ["D6", "D2", "I6", "X1", "A1", "P5", "T2", "D1", "M5", "P4"],
    "SZ 8P": ["D2", "K'9", "K8", "N4", "L1", "M5", "K7", "N5", "D1", "S4"],
}

FRAG_SHAFT = {  # shaft -> votes/ch, as reported by the R run
    "D": 4.83, "I": 2.00, "M": 1.38, "T": 1.33, "F": 1.30, "S": 1.25,
    "P": 1.10, "N": 1.00, "X": 1.00, "B": 0.88, "K": 0.60, "A": 0.50,
    "G'": 0.50, "L": 0.50,
}

CLINICAL = {"I": "EEG ONSET", "A": "EEG ONSET", "S": "early spread",
            "P": "early spread", "N": "early spread", "K": "early spread",
            "L": "early spread"}

In [5]:
print(f"{'seizure':8}  {'EI top 10':52}  fragility top 10")
for label in sorted(ei_full):
    print(f"{label:8}  {', '.join(ei_full[label][:10]):52}  {', '.join(FRAG_TOP[label])}")

# Cutoff-free: which shaft wins each seizure outright.
print("\n#1 contact per seizure")
print("  EI:        ", ", ".join(ei_full[L][0] for L in sorted(ei_full)))
print("  fragility: ", ", ".join(FRAG_TOP[L][0] for L in sorted(FRAG_TOP)))
for name, src in (("EI", ei_full), ("fragility", FRAG_TOP)):
    wins = defaultdict(int)
    for L in src:
        wins[shaft_of(src[L][0])] += 1
    ordered = sorted(wins.items(), key=lambda kv: -kv[1])
    print(f"  {name:10s} shaft wins: " + ", ".join(f"{s}:{n}/8" for s, n in ordered))

seizure   EI top 10                                             fragility top 10
SZ 1P     G'10, L'10, M'8, M'9, X12, D3, P1, G2, D5, D4         X12, P1, D6, L'11, D2, L'8, M5, D1, L'7, P6
SZ 2P     M'10, G3, I6, A1, X2, X3, A10, L'4, I5, L'1           D2, D1, S4, L8, D3, B4, I3, F10, X12, F6
SZ 3P     X'2, L'5, G1, M'1, M'10, M'4, G'5, X'1, P1, X12       S4, D2, P6, T3, D1, M5, P1, T2, X12, K7
SZ 4P     I6, L1, K6, N1, K5, K7, M2, M4, M3, K'12              D6, D2, D1, F5, M5, A1, K8, M8, F6, P6
SZ 5P     I1, X'12, I2, A2, I3, A3, A4, I6, A1, A5              D1, D2, D6, X'12, I1, T3, T2, I2, A1, X9
SZ 6P     L'6, L'7, G'6, G'5, L'4, L'5, L'8, I6, L'3, M'1       D6, I1, M5, X'12, I2, B4, I3, I6, M'10, X'8
SZ 7P     I6, A1, B4, B5, B3, L1, N1, A7, A6, D5                D6, D2, I6, X1, A1, P5, T2, D1, M5, P4
SZ 8P     S2, D5, D3, D4, L'8, D2, G'3, N5, N4, N3              D2, K'9, K8, N4, L1, M5, K7, N5, D1, S4

#1 contact per seizure
  EI:         G'10, M'10, X'2, I6, I1, L'6, I6, S2
  fr

## Cutoff sweep

Votes = contacts in a seizure's top N, summed over the 8 seizures, divided by the
shaft's contact count (so a 12-contact shaft doesn't beat a 6-contact one on size).
`chance` is what a shaft scores if the top N were drawn at random
(`8 * N / n_contacts`) — a shaft only means something well above it.

In [6]:
def shaft_votes(top_n):
    votes = defaultdict(int)
    for ranking in ei_full.values():
        for c in ranking[:top_n]:
            votes[shaft_of(c)] += 1
    return {s: v / shaft_size[s] for s, v in votes.items()}


for top_n in (5, 10, 20, 30):
    norm = shaft_votes(top_n)
    chance = 8 * top_n / N_CONTACTS
    ranked = sorted(norm, key=lambda s: -norm[s])[:8]
    print(f"top {top_n:2d} (chance {chance:.2f}):  " +
          "  ".join(f"{s}:{norm[s]:.2f}" for s in ranked))

print(f"\n{'shaft':>6} {'n':>4} {'EI@10':>7} {'EI@20':>7} {'frag':>7}  clinical")
n10, n20 = shaft_votes(10), shaft_votes(20)
for sh in sorted(set(n20) | set(FRAG_SHAFT), key=lambda s: -n20.get(s, 0)):
    frag = FRAG_SHAFT.get(sh)
    print(f"{sh:>6} {shaft_size.get(sh, 0):4d} {n10.get(sh, 0):7.2f} {n20.get(sh, 0):7.2f} "
          f"{('%.2f' % frag) if frag is not None else '-':>7}  {CLINICAL.get(sh, '')}")

top  5 (chance 0.22):  I:1.00  L':0.50  M':0.50  D:0.50  B:0.38  G':0.30  A:0.30  G:0.20
top 10 (chance 0.43):  I:1.50  D:1.33  A:1.00  L':0.92  M':0.70  N:0.62  G':0.50  M:0.38
top 20 (chance 0.87):  D:2.67  I:1.83  L':1.50  B:1.38  G':1.20  M':1.20  A:1.20  X:1.00
top 30 (chance 1.30):  D:3.00  I:2.67  B:2.25  M':2.10  L':2.08  A:1.80  K:1.70  G':1.50

 shaft    n   EI@10   EI@20    frag  clinical
     D    6    1.33    2.67    4.83  
     I    6    1.50    1.83    2.00  EEG ONSET
    L'   12    0.92    1.50       -  
     B    8    0.38    1.38    0.88  
    G'   10    0.50    1.20    0.50  
    M'   10    0.70    1.20       -  
     A   10    1.00    1.20    0.50  EEG ONSET
     N    8    0.62    1.00    1.00  early spread
     X   12    0.33    1.00    1.00  
     K   10    0.30    0.90    0.60  early spread
     L   10    0.20    0.90    0.50  early spread
     M    8    0.38    0.75    1.38  
    X'   12    0.25    0.50       -  
    K'   12    0.08    0.42       -  
     G   10

In [7]:
# Lateralisation: primed shafts are the contralateral side. Their share of votes
# against their share of the implant -- equal means no lateralisation, whatever
# any single seizure looks like.
for top_n in (10, 20):
    votes = defaultdict(int)
    for ranking in ei_full.values():
        for c in ranking[:top_n]:
            votes[shaft_of(c)] += 1
    primed = sum(v for s, v in votes.items() if s.endswith("'"))
    total = sum(votes.values())
    print(f"top {top_n}: primed {primed}/{total} = {primed / total:.0%}")

n_primed = sum(n for s, n in shaft_size.items() if s.endswith("'"))
print(f"implant: primed {n_primed}/{N_CONTACTS} = {n_primed / N_CONTACTS:.0%}\n")

for label in sorted(ei_full):
    n = sum(1 for c in ei_full[label][:10] if shaft_of(c).endswith("'"))
    print(f"  {label}: {n}/10 primed in EI top 10")

top 10: primed 27/80 = 34%
top 20: primed 53/160 = 33%
implant: primed 56/184 = 30%

  SZ 1P: 4/10 primed in EI top 10
  SZ 2P: 3/10 primed in EI top 10
  SZ 3P: 7/10 primed in EI top 10
  SZ 4P: 1/10 primed in EI top 10
  SZ 5P: 1/10 primed in EI top 10
  SZ 6P: 9/10 primed in EI top 10
  SZ 7P: 0/10 primed in EI top 10
  SZ 8P: 2/10 primed in EI top 10


In [16]:
ei_full["SZ 1P"]

["G'10",
 "L'10",
 "M'8",
 "M'9",
 'X12',
 'D3',
 'P1',
 'G2',
 'D5',
 'D4',
 'D2',
 'B2',
 "X'12",
 'D1',
 'B3',
 'I1',
 'G3',
 'B1',
 'I2',
 "L'1",
 'A10',
 'A3',
 'A7',
 'A2',
 'I3',
 'B4',
 'D6',
 'X1',
 'F2',
 'M2',
 'A5',
 "G'6",
 'B8',
 'A6',
 'P4',
 'A4',
 'M3',
 "G'7",
 "G'5",
 'P2',
 "X'3",
 'B5',
 "X'2",
 'X2',
 "X'8",
 'Q1',
 "L'12",
 'M7',
 'G6',
 'G1',
 'F1',
 'G5',
 'M1',
 "K'1",
 "X'4",
 'N2',
 "L'11",
 'X3',
 "X'11",
 'M6',
 'N5',
 'G4',
 "X'10",
 'L7',
 'T6',
 'N8',
 'M5',
 "L'8",
 'S2',
 "X'5",
 'N4',
 "G'2",
 'K1',
 'L5',
 'M8',
 'N1',
 "L'9",
 "K'12",
 "M'10",
 "G'3",
 'F4',
 'X5',
 'K8',
 'P6',
 'P9',
 "G'9",
 "L'7",
 'X4',
 "G'8",
 "G'4",
 'K7',
 'S1',
 'T1',
 "G'1",
 'P5',
 'F5',
 'P3',
 'F3',
 'L2',
 'G9',
 'G10',
 'L4',
 'P8',
 'G8',
 'M4',
 'P7',
 "X'1",
 "L'2",
 'P10',
 "X'6",
 'L6',
 "M'2",
 'F6',
 "M'1",
 'F7',
 'F8',
 'G7',
 'Q6',
 "K'3",
 "M'3",
 'Q2',
 "X'9",
 "K'2",
 'F9',
 "L'6",
 'K9',
 'Q4',
 'Q5',
 'L3',
 'K3',
 'S4',
 'T4',
 "M'5",
 'L8',
 'S7',
 